# Assignment 10


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

gray_img = cv2.imread('Sunflower_from_Silesia2.jpg', cv2.IMREAD_GRAYSCALE)


## Question 1: Adaptive Median Filter


In [ ]:
def add_salt_and_pepper_noise(image, salt_prob=0.15, pepper_prob=0.15):
    noisy = image.copy()
    num_salt = np.ceil(salt_prob * image.size)
    coords = [np.random.randint(0, i - 1, int(num_salt)) for i in image.shape]
    noisy[tuple(coords)] = 255

    num_pepper = np.ceil(pepper_prob * image.size)
    coords = [np.random.randint(0, i - 1, int(num_pepper)) for i in image.shape]
    noisy[tuple(coords)] = 0
    return noisy

np.random.seed(42)
noisy_sp = add_salt_and_pepper_noise(gray_img, salt_prob=0.15, pepper_prob=0.15)


In [ ]:
try:
    smin_in = input('Enter S_min [Default 3]: ')
    smax_in = input('Enter S_max [Default 7]: ')

    s_min = int(smin_in) if smin_in.strip() != '' else 3
    s_max = int(smax_in) if smax_in.strip() != '' else 7
    if s_min % 2 == 0: s_min += 1
    if s_max % 2 == 0: s_max += 1
    if s_max < s_min: s_max = s_min
except (ValueError, EOFError):
    s_min = 3
    s_max = 7

def adaptive_median_filter(image, s_min=3, s_max=7):
    h, w = image.shape
    pad = s_max // 2
    padded = np.pad(image, pad, mode='edge')
    output = np.zeros_like(image)

    for i in range(h):
        for j in range(w):
            z_xy = int(image[i, j])
            current_s = s_min

            while current_s <= s_max:
                half_s = current_s // 2
                r_start = (i + pad) - half_s
                r_end = (i + pad) + half_s + 1
                c_start = (j + pad) - half_s
                c_end = (j + pad) + half_s + 1

                window = padded[r_start:r_end, c_start:c_end]
                z_min = int(np.min(window))
                z_max = int(np.max(window))
                z_med = int(np.median(window))

                a1 = z_med - z_min
                a2 = z_med - z_max

                if a1 > 0 and a2 < 0:
                    b1 = z_xy - z_min
                    b2 = z_xy - z_max
                    if b1 > 0 and b2 < 0:
                        output[i, j] = z_xy
                    else:
                        output[i, j] = z_med
                    break
                else:
                    current_s += 2
                    if current_s > s_max:
                        output[i, j] = z_med
                        break

    return output

standard_median = cv2.medianBlur(noisy_sp, s_min)
adaptive_median = adaptive_median_filter(noisy_sp, s_min=s_min, s_max=s_max)

def calc_psnr(orig, filtered):
    mse = np.mean((orig.astype(float) - filtered.astype(float)) ** 2)
    return 10.0 * np.log10((255.0 ** 2) / mse) if mse > 0 else float('inf')

psnr_noisy = calc_psnr(gray_img, noisy_sp)
psnr_std = calc_psnr(gray_img, standard_median)
psnr_amf = calc_psnr(gray_img, adaptive_median)

print(f'Noisy Image PSNR:            {psnr_noisy:.2f} dB')
print(f'Standard Median PSNR:        {psnr_std:.2f} dB')
print(f'Adaptive Median Filter PSNR: {psnr_amf:.2f} dB')

plt.figure(figsize=(16, 5))

plt.subplot(1, 4, 1)
plt.title(f'Noisy Image (30% S&P)\nPSNR: {psnr_noisy:.2f} dB')
plt.imshow(noisy_sp, cmap='gray')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.title(f'Standard Median ({s_min}x{s_min})\nPSNR: {psnr_std:.2f} dB')
plt.imshow(standard_median, cmap='gray')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.title(f'Adaptive Median ({s_min}\u2192{s_max})\nPSNR: {psnr_amf:.2f} dB')
plt.imshow(adaptive_median, cmap='gray')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.title('Original Clean Image')
plt.imshow(gray_img, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()


## Question 2: Otsu Global Thresholding


In [ ]:
def manual_otsu_threshold(image):
    hist, _ = np.histogram(image.ravel(), bins=256, range=[0, 256])
    total_pixels = image.size
    p = hist / total_pixels

    omega = np.cumsum(p)
    mu = np.cumsum(np.arange(256) * p)
    mu_global = mu[-1]

    sigma_b_squared = np.zeros(256)
    for k in range(256):
        w1 = omega[k]
        w2 = 1.0 - w1
        if w1 > 0 and w2 > 0:
            m1 = mu[k] / w1
            m2 = (mu_global - mu[k]) / w2
            sigma_b_squared[k] = w1 * w2 * ((m1 - m2) ** 2)

    optimal_threshold = int(np.argmax(sigma_b_squared))
    segmented = np.where(image > optimal_threshold, 255, 0).astype(np.uint8)
    return optimal_threshold, segmented, sigma_b_squared, hist

opt_thresh, otsu_segmented, sigma_b2, hist = manual_otsu_threshold(gray_img)
cv_thresh_val, cv_otsu_segmented = cv2.threshold(gray_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

print(f'Manual Otsu Optimal Threshold: {opt_thresh}')
print(f'OpenCV Otsu Optimal Threshold: {cv_thresh_val}')

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title('Original Grayscale Image')
plt.imshow(gray_img, cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title(f'Otsu Segmentation (Threshold = {opt_thresh})')
plt.imshow(otsu_segmented, cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.plot(range(256), hist / hist.max(), color='gray', alpha=0.6, label='Normalized Histogram')
plt.plot(range(256), sigma_b2 / sigma_b2.max(), color='blue', lw=2, label='Between-Class Variance \u03c3_b\u00b2')
plt.axvline(opt_thresh, color='red', linestyle='--', lw=2, label=f'Threshold k*={opt_thresh}')
plt.title("Otsu's Criterion Function")
plt.xlabel('Intensity Level k')
plt.ylabel('Normalized Amplitude')
plt.legend()

plt.tight_layout()
plt.show()
